In [7]:
import pandas as pd
import numpy as np
import h3
import glob
import os
import heapq
import math
import requests
from collections import defaultdict

# ─── Helper Functions ─────────────────────────────────────────────
def bearing(lat1, lon1, lat2, lon2):
    dlon = math.radians(lon2 - lon1)
    x = math.sin(dlon) * math.cos(math.radians(lat2))
    y = (math.cos(math.radians(lat1)) * math.sin(math.radians(lat2)) -
         math.sin(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.cos(dlon))
    return math.degrees(math.atan2(x, y)) % 360

def direction_compatible(path_a, path_b, threshold=90):
    bear_a = bearing(*path_a[0], *path_a[-1])
    bear_b = bearing(*path_b[0], *path_b[-1])
    diff = abs(bear_a - bear_b)
    diff = min(diff, 360 - diff)
    return diff < threshold

def path_similarity(h3_path_1, h3_path_2, coords_1, coords_2, k=2):
    if not direction_compatible(coords_1, coords_2, threshold=90):
        return 0.0
    set_1 = set()
    for h in h3_path_1: set_1.update(h3.grid_disk(h, k))
    set_2 = set()
    for h in h3_path_2: set_2.update(h3.grid_disk(h, k))
    cov_1 = sum(1 for h in h3_path_1 if h in set_2) / len(h3_path_1)
    cov_2 = sum(1 for h in h3_path_2 if h in set_1) / len(h3_path_2)
    return min(cov_1, cov_2)

def check_osrm_detour(A_curr, B_curr, A_end, B_end, threshold=30.0):
    def get_route_legs(coords):
        coords_str = ";".join([f"{lon},{lat}" for lat, lon in coords])
        url = f"http://127.0.0.1:5000/route/v1/driving/{coords_str}?overview=false"
        try:
            res = requests.get(url).json()
            if res.get("code") == "Ok": return [leg["distance"] for leg in res["routes"][0]["legs"]]
        except Exception: pass
        return None

    legs1 = get_route_legs([A_curr, B_curr, A_end, B_end])
    legs2 = get_route_legs([A_curr, B_curr, B_end, A_end])
    legsA = get_route_legs([A_curr, A_end])
    if not legs1 or not legs2 or not legsA: return False, 0, 0, None , 0, 0, 0, 0, 0
    direct_A = legsA[0]
    direct_B = legs2[1]  
    if direct_A <= 0 or direct_B <= 0: return False, 0, 0, None , 0, 0, 0, 0, 0
    dist1, dist2 = sum(legs1), sum(legs2)

    path ="ABAB"
    destins_dis=0
    if dist1 < dist2:
        detour_A = ((legs1[0] + legs1[1]) - direct_A) / direct_A * 100
        detour_B = ((legs1[1] + legs1[2]) - direct_B) / direct_B * 100
        destins_dis=legs1[2]
    else:
        detour_A = ((legs2[0] + legs2[1] + legs2[2]) - direct_A) / direct_A * 100
        detour_B = 0.0 
        destins_dis=legs2[2]
        path ="ABBA"
    return detour_A <= threshold and detour_B <= threshold, direct_A, direct_B, path , min(dist1,dist2), detour_A, detour_B, legs1[0], destins_dis

def _unique_ordered(series):
    seen = set()
    return [v for v in series if not (v in seen or seen.add(v))]


# ─── File Discovery ───────────────────────────────────────────────
POOL_DIR = "./poolingdata/2025-08-04"
parquet_files = sorted(glob.glob(os.path.join(POOL_DIR, "*.parquet")))
print(f"Found {len(parquet_files)} parquet files in {POOL_DIR}\n")

# ─── PASS 1: Build Metadata Dictionary (Low RAM) ──────────────────
print("--- Phase 1: Building Full Trajectory Meta ---")

import pickle

load_path = "./data2/ride_meta_sim_08_04.pkl"

with open(load_path, 'rb') as f:
    ride_meta_sim = pickle.load(f)

print("Loaded ride_meta_sim into memory...")
print(len(ride_meta_sim))



# ─── PASS 2: File-by-File Sweep-Line (Low RAM) ────────────────────
print("--- Phase 2: Sweep-Line Streaming ---")

# Global State trackers (persists across files)
active_heap    = []                
active_rides   = {}                
removed_rids   = set()             
hex_to_rids    = defaultdict(set)  
batched_rids   = set()
tried_rids     = {}
seq            = 0
osrm_count     = 0
max_active     = 0
total_evicted  = 0
batchid        = 0
 
# Instead of massive arrays (best_sim_scores = [np.nan] * len(df)), 
# we append successful matches to a list. Huge memory saver.
matched_batches = [] 

for file_idx, fp in enumerate(parquet_files):
    print(f"\nProcessing File {file_idx+1}/{len(parquet_files)}: {os.path.basename(fp)}")
    
    # Load slim chunk
    slim_cols = ["rid", "ts", "lat", "lon", "rideStatus"]
    df = pd.read_parquet(fp, columns=slim_cols, engine="pyarrow")
    df["lat"] = pd.to_numeric(df["lat"], errors="coerce")
    df["lon"] = pd.to_numeric(df["lon"], errors="coerce")
    df = df.dropna(subset=["lat", "lon"])
    df["ts"] = pd.to_datetime(df["ts"])
    df["curr_hex"] = [h3.latlng_to_cell(lat, lon, 9) for lat, lon in zip(df["lat"], df["lon"])]
    df["rideStatus"] = df["rideStatus"].astype("category")
    
    # Sort chronologically within the chunk
    df = df.sort_values("ts").reset_index(drop=True)
    columns = df.columns.tolist()
    
    for i, row in df.iterrows():
        current_ts    = row['ts']
        rid           = row['rid']
        curr_hex      = row['curr_hex']
        current_ts_ns = current_ts.value
        status        = row['rideStatus']
        

        meta = ride_meta_sim.get(rid)
        if not meta: continue

        end_time      = meta['trip_end_time']
        end_time_ns   = end_time.value
        start_time      = meta['trip_start_time']
        start_time_ns   = start_time.value
        B_trip_start_lat  = meta['trip_start_lat']
        B_trip_start_lon  = meta['trip_start_lon']
        B_trip_end_lat  = meta['trip_end_lat']
        B_trip_end_lon  = meta['trip_end_lon']

        # Clean up expired rides
        while active_heap and active_heap[0][0] <= current_ts_ns:
            _, _, evict_rid = heapq.heappop(active_heap)
            if evict_rid in active_rides:
                evict_hex = active_rides[evict_rid].get('curr_hex')
                if evict_hex and evict_rid in hex_to_rids[evict_hex]:
                    hex_to_rids[evict_hex].discard(evict_rid)
                    if not hex_to_rids[evict_hex]: del hex_to_rids[evict_hex]
                del active_rides[evict_rid]
                total_evicted += 1

        is_new_ride = rid not in active_rides

        # Update existing ride location
        if not is_new_ride:
            old_hex = active_rides[rid].get('curr_hex')
            if old_hex != curr_hex:
                if old_hex and rid in hex_to_rids[old_hex]:
                    hex_to_rids[old_hex].discard(rid)
                    if not hex_to_rids[old_hex]: del hex_to_rids[old_hex]
                hex_to_rids[curr_hex].add(rid)
            removed_rids.add(rid)

            ride_data = row.to_dict() # Cleaner than dict comprehension on columns
            ride_data.update({
                'trip_start_lat': meta['trip_start_lat'],
                'trip_start_lon': meta['trip_start_lon'],
                'trip_end_lat': meta['trip_end_lat'],
                'trip_end_lon': meta['trip_end_lon'],
                'journey_hexes': meta['journey_hexes']
            })
            active_rides[rid] = ride_data
            # if rid == "b7b7d96b-c71e-45ae-9a67-27f90ccefad9":
            #     print(rid, active_rides["b7b7d96b-c71e-45ae-9a67-27f90ccefad9"])

        is_not_batched_ride = rid not in batched_rids

        is_tried_ride = rid in tried_rids

        if is_tried_ride and current_ts_ns - tried_rids[rid] > 15 * 1e9:
            is_tried_ride = False

        # Process Pickups
        if is_new_ride and not is_tried_ride and is_not_batched_ride and status == "ON_PICKUP":
            tried_rids[rid] = current_ts_ns
            
            curr_hex = h3.latlng_to_cell(B_trip_start_lat, B_trip_start_lon, 9)

            neighbor_rids = []
            for h in h3.grid_disk(curr_hex, 2):
                if h in hex_to_rids: neighbor_rids.extend(hex_to_rids[h])

            B_hexes = meta['journey_hexes']
            b_start_idx = B_hexes.index(curr_hex) if curr_hex in B_hexes else 0
                
            rem_B_hexes = B_hexes[b_start_idx:]
            rem_B_coords = [(B_trip_start_lat, B_trip_start_lon), (B_trip_end_lat, B_trip_end_lon)] # if len(B_coords) > 0 else []

            best_score = 0.0
            best_rid = None
            A_curr_hex = None

            if neighbor_rids and rem_B_hexes and len(rem_B_coords) >= 2:
                for n_rid in neighbor_rids:
                    if n_rid in batched_rids: continue
                    ar = active_rides[n_rid]
                    
                    A_hexes = ar['journey_hexes']
                    A_curr_hex = ar['curr_hex']
                    A_trip_end_lat  = ar['trip_end_lat']
                    A_trip_end_lon  = ar['trip_end_lon']
                    
                    a_start_idx = A_hexes.index(A_curr_hex) if A_curr_hex in A_hexes else 0
                        
                    rem_A_hexes = A_hexes[a_start_idx:]
                    rem_A_coords = [(ar['lat'], ar['lon']), (A_trip_end_lat, A_trip_end_lon)] # if len(A_coords) > 0 else []

                    if rem_A_hexes and len(rem_A_coords) >= 2:
                        sim = path_similarity(rem_A_hexes, rem_B_hexes, rem_A_coords, rem_B_coords, k=2)
                        if sim > best_score:
                            best_score = sim
                            best_rid = n_rid
                            best_rid_hex = A_curr_hex 
                            
            if best_score > 0.1 and best_rid is not None:
                ar = active_rides[best_rid]
                A_curr = (ar['lat'], ar['lon'])
                B_curr = (row['lat'], row['lon'])
                A_end = (ar['trip_end_lat'], ar['trip_end_lon'])
                B_end = (meta['trip_end_lat'], meta['trip_end_lon'])
                
                is_batchable, direct_A, direct_B, path, path_dis, detour_A, detour_B, pickup_dis, destins_dis  = check_osrm_detour(A_curr, B_curr, A_end, B_end, threshold=30.0)
                osrm_count += 1
                print(i, osrm_count, is_batchable)
                if is_batchable:
                    batched_rids.add(best_rid)
                    batched_rids.add(rid)
                    batchid+=1
                    # Save the match immediately
                    matched_batches.append({
                        'batchid': batchid,
                        'ts': current_ts,
                        'rid': rid,
                        'matched_rid': best_rid,
                        'rid_hexB': curr_hex,
                        'rid_hexA': A_curr_hex,
                        'matched_sim_score': best_score,
                        'directA': direct_A,
                        'directB': direct_B,
                        'path': path,
                        'pathdis': path_dis,
                        'detourA': detour_A,
                        'detourB': detour_B,
                        'pickup_dis': pickup_dis,
                        'destins_dis': destins_dis
                    })

        # Process Active Rides
        if is_new_ride and status == "ON_RIDE":
            hex_to_rids[curr_hex].add(rid)

            ride_data = row.to_dict()
            ride_data.update({
                'trip_start_lat': meta['trip_start_lat'],
                'trip_start_lon': meta['trip_start_lon'],
                'trip_end_lat': meta['trip_end_lat'],
                'trip_end_lon': meta['trip_end_lon'],
                'journey_hexes': meta['journey_hexes']
            })
        
            active_rides[rid] = ride_data
            seq += 1
            heapq.heappush(active_heap, (end_time_ns, seq, rid))

        n_active = len(active_rides)
        if n_active > max_active: max_active = n_active
            
    # Free the DataFrame chunk before looping to the next file
    del df

# ─── Post-Run Summary ─────────────────────────────────────────────
print(f"\n✓ Sweep-line Streaming Complete")
print(f"  Peak active rides       : {max_active}")
print(f"  Total evicted           : {total_evicted}")
print(f"  Still active (end)      : {len(active_rides)}")
print(f"  Total Batched Rides     : {len(batched_rids)}")

batchable_df = pd.DataFrame(matched_batches)
print(f"  Batchable matches found : {len(batchable_df)}")

if not batchable_df.empty:
    print("\nSample batchable matches:")
    print(batchable_df.head(5).to_string(index=False))

batchable_df.to_parquet(
    "./results2/pooled08043.parquet",
    engine="pyarrow",
    compression="snappy",   # fast, default choice
    index=False
)

Found 51 parquet files in ./poolingdata/2025-08-04

--- Phase 1: Building Full Trajectory Meta ---
Loaded ride_meta_sim into memory...
76941
--- Phase 2: Sweep-Line Streaming ---

Processing File 1/51: part-00000-c3370d9f-57bf-490b-9742-dc6464b14d3c-c000.snappy.parquet
3127 1 False
3172 2 False
3701 3 False
3754 4 False
9083 5 False
9167 6 False
9240 7 False
9362 8 False
9455 9 False
9561 10 False
9677 11 False
9774 12 False
9891 13 False
10010 14 False
16007 15 False
16145 16 False
16335 17 True
17457 18 False
17537 19 False
17662 20 False
17829 21 False
17954 22 False
18024 23 False
18126 24 False
18197 25 False
18308 26 False
18379 27 False
18490 28 False
18651 29 False
18785 30 False
18922 31 False
19038 32 False
19276 33 False
19456 34 False
19647 35 False
19833 36 False
20059 37 False
20192 38 False
20420 39 False
25525 40 True
27353 41 False
27579 42 False
27790 43 False
28060 44 True
28073 45 False
28316 46 False
30363 47 False
30403 48 False
30600 49 False
30823 50 False
31042

In [3]:
len(matched_batches)

6516